Second model, second generative family. Drax is discrete flow matching; Whisfusion is masked diffusion.

**GPU T4 x2**, Internet on, attach `prepare-data-for-word-reranker`. Run the cells one at a time — the install is the risky part.

In [ ]:
REPO_URL = "https://github.com/Splestule/candidate_reranker.git"
BRANCH = "main"

In [ ]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K

COMMIT = K.sync(REPO_URL, BRANCH)
K.gpu_info()
env = K.prepare(COMMIT)

Drax pins `torch==2.7.0`. Kaggle ships a newer one and downgrading it breaks CUDA, so this installs **without dependencies** on purpose.

In [ ]:
DRAX = Path("/kaggle/working/drax")
if not DRAX.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/aiola-lab/drax.git", str(DRAX)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(DRAX), "--no-deps"],
               check=True)
print("installed without deps")

Probe the import. Whatever it reports missing goes into the next cell — fail fast, one dependency at a time.

In [ ]:
import importlib, traceback

try:
    importlib.import_module("drax")
    print("drax imports cleanly")
except Exception:
    traceback.print_exc()
    print("\n^ install what it names, then rerun this cell")

In [ ]:
MISSING = []          # e.g. ["einops", "omegaconf"]

if MISSING:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *MISSING], check=True)
    print("installed:", MISSING)
else:
    print("nothing to install yet")

One utterance, k = 4, to prove it runs on a T4 at all and that the candidates actually differ. If this is out of memory, drop `k`; if it is slow, drop `sampling_steps`.

In [ ]:
import torch
from drax import Transcriber

asr = Transcriber(model_path="aiola/drax-v1")

import data as dataio, itertools
u = next(itertools.islice(dataio.iter_utterances("librispeech", env.librispeech / "test-clean"), 1))
print("REF", u.text, "\n")

res = asr.transcribe([u.audio_path] * 4, language=["en"] * 4,
                     sampling_steps=8, temperature=0.6)
for i, r in enumerate(res):
    print(i, r.transcript)

print(f"\nunikatnich: {len({r.transcript for r in res})} ze 4")
print(f"spicka pameti: {torch.cuda.max_memory_allocated() / 2**20:.0f} MB")

Full smoke test: 40 utterances up to 15 s, four temperatures. The candidates only diverge if the temperature is high enough, and run 1 warned that random diversity can hurt — so this sweeps it rather than guessing.

In [ ]:
rc = K.run(env, "drax_smoke.py",
           "--librispeech", env.librispeech / "test-clean",
           "--out_dir", env.results,
           "--n_utts", 40, "--k", 10,
           "--sampling_steps", 8,
           "--temperatures", "0.1,0.3,0.6,1.0")
assert rc == 0, f"drax_smoke exit code {rc}, see the output above"